In [2]:
import pandas as pd

# ------------------------------------------
# 1. File paths
# ------------------------------------------
csv1 = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Combined_Castello_with_Themes.csv"
csv2 = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Combined_Castello.csv"

# ------------------------------------------
# 2. Load datasets
# ------------------------------------------
df1 = pd.read_csv(csv1)   # contains theme_labels
df2 = pd.read_csv(csv2)   # base file

# ------------------------------------------
# 3. Keep only url_s + theme_labels from df1
# ------------------------------------------
df1_small = df1[['url_s', 'theme_labels']]

# ------------------------------------------
# 4. Merge using url_s
# left merge ensures df2 remains the base dataset
# ------------------------------------------
merged = df2.merge(df1_small, on='url_s', how='left')

# ------------------------------------------
# 5. Export output
# ------------------------------------------
output_path = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Combined_Castello_Merged.csv"
merged.to_csv(output_path, index=False)

print("Merged CSV saved to:", output_path)


Merged CSV saved to: C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Combined_Castello_Merged.csv


In [7]:
import pandas as pd
import os

# ---------------------------------------
# Input & Output Paths
# ---------------------------------------
input_csv = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Combined_Castello_Merged.csv"
output_folder = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories"

os.makedirs(output_folder, exist_ok=True)

# ---------------------------------------
# Load CSV
# ---------------------------------------
df = pd.read_csv(input_csv)
df["theme_labels"] = df["theme_labels"].fillna("")

# ---------------------------------------
# Correct Category Spellings
# ---------------------------------------
categories = [
    "Tourism",
    "Water & mobility",
    "Culture & heritage",
    "Public space life",
    "Nature & environment",
    "Architecture",
    "Commerce",
    "Events"
]

# ---------------------------------------
# Function to check if category exists in row
# Ensures exact matching inside comma-separated list
# ---------------------------------------
def row_has_category(row_labels, category):
    # split by comma, trim spaces, lower everything
    labels = [x.strip().lower() for x in row_labels.split(",")]
    return category.lower() in labels


# ---------------------------------------
# Process each category
# ---------------------------------------
counts = {}

for cat in categories:
    filtered_df = df[df["theme_labels"].apply(lambda x: row_has_category(x, cat))]

    output_name = cat.replace("&", "and").replace(" ", "_")
    output_path = os.path.join(output_folder, f"{output_name}.csv")

    filtered_df.to_csv(output_path, index=False)

    counts[cat] = len(filtered_df)

# ---------------------------------------
# Print Summary
# ---------------------------------------
print("\n=== CATEGORY COUNTS ===")
for cat, count in counts.items():
    print(f"{cat}: {count}")



=== CATEGORY COUNTS ===
Tourism: 2928
Water & mobility: 1360
Culture & heritage: 2831
Public space life: 886
Nature & environment: 444
Architecture: 2719
Commerce: 2611
Events: 1128


In [5]:
pip install plotly


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.9 MB 6.1 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/9.9 MB 6.0 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.9 MB 6.1 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.9 MB 6.1 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.9 MB 5.9 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/9.9 MB 5.8 MB/s eta 0:00:01
   --------------------------------- ------ 8.4/9.9 MB 5.9 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 5.9 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 5.8 MB/s  0:00:01

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ----------

In [6]:
import pandas as pd
from collections import Counter
import plotly.graph_objects as go

# ----------------------------------------------------------
# 1. Load CSV
# ----------------------------------------------------------
csv_path = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Combined_Castello_Merged.csv"
df = pd.read_csv(csv_path)

# ----------------------------------------------------------
# 2. Extract and count theme labels
# ----------------------------------------------------------
all_labels = []

for labels in df["theme_labels"].dropna():
    for label in labels.split(","):
        clean_label = label.strip()
        if clean_label:
            all_labels.append(clean_label)

counts = Counter(all_labels)

# ----------------------------------------------------------
# 3. Remove categories you want to ignore
# ----------------------------------------------------------
ignore = ["Archietcture", "nature & enviornment"]
for k in ignore:
    if k in counts:
        del counts[k]

# Keep only relevant categories
categories = list(counts.keys())
values = list(counts.values())

print("Category Counts:")
for c, v in counts.items():
    print(f"{c}: {v}")

# ----------------------------------------------------------
# 4. Build Sankey-Style Proportional Visualization
# ----------------------------------------------------------
# Single source node "All Data"
source_label = ["All Images"]
source_index = 0

labels = ["All Images"] + categories
sources = []
targets = []
sizes = []

for i, cat in enumerate(categories):
    sources.append(0)                # All Images → category
    targets.append(i + 1)            # target index offset by +1
    sizes.append(counts[cat])

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels
    ),
    link=dict(
        source=sources,
        target=targets,
        value=sizes,
        color="rgba(180, 50, 200, 0.4)"
    )
)])

fig.update_layout(
    title_text="Category Distribution — Proportional Flow Diagram",
    font_size=12,
    width=1200,
    height=800
)

# ----------------------------------------------------------
# 5. Save result as HTML (open in Chrome)
# ----------------------------------------------------------
output_path = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\category_flow.html"
fig.write_html(output_path)

print("\nVisualization saved to:")
print(output_path)


Category Counts:
Tourism: 2928
Water & mobility: 1360
Culture & heritage: 2831
Nature & environment: 444
Architecture: 2719
Public space life: 886
Commerce: 2611
Events: 1128

Visualization saved to:
C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\category_flow.html


In [17]:
import pandas as pd

# -----------------------------
# File paths
# -----------------------------
input_csv = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\csv\Architecture.csv"
output_csv = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\csv\Architecture_cleaned.csv"

# -----------------------------
# Load CSV
# -----------------------------
df = pd.read_csv(input_csv)

# -----------------------------
# Replace commas in theme_labels
# -----------------------------
if "theme_labels" in df.columns:
    df["theme_labels"] = df["theme_labels"].fillna("").astype(str)
    df["theme_labels"] = df["theme_labels"].str.replace(",", "_")

# -----------------------------
# Replace empty cells in all other columns with NA
# -----------------------------
df = df.replace("", "NA")     # empty strings → NA
df = df.fillna("NA")          # actual NaN → NA

# -----------------------------
# Save output
# -----------------------------
df.to_csv(output_csv, index=False)

print("✔ Cleaning complete!")
print("Saved to:", output_csv)


✔ Cleaning complete!
Saved to: C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\csv\Architecture_cleaned.csv


In [21]:
import pandas as pd

# Input and output file paths
input_file = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\csv\Architecture_cleaned.csv"
output_file = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\csv\Architecture_cleaned_no_commas.csv"

# Read the CSV
df = pd.read_csv(input_file)

# Columns to clean
columns_to_clean = ['title', 'tags', 'one_comment']

# Replace commas with " - " and line breaks with a space
for col in columns_to_clean:
    if col in df.columns:
        df[col] = df[col].astype(str) \
                            .str.replace(',', ' - ') \
                            .str.replace('\n', ' ') \
                            .str.replace('\r', ' ')

# Export the cleaned CSV
df.to_csv(output_file, index=False, lineterminator='\n')

print("CSV cleaned, line breaks fixed, and exported successfully!")


CSV cleaned, line breaks fixed, and exported successfully!


In [28]:
import pandas as pd

# -----------------------------
# File paths
# -----------------------------
input_csv = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\csv\Water_and_mobility.csv"
output_csv = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\csv\Water_and_mobility_cleaned_final.csv"

# -----------------------------
# Load CSV
# -----------------------------
df = pd.read_csv(input_csv)

# -----------------------------
# Clean theme_labels
# -----------------------------
if "theme_labels" in df.columns:
    df["theme_labels"] = df["theme_labels"].fillna("").astype(str)
    df["theme_labels"] = df["theme_labels"].str.replace(",", "_")

# -----------------------------
# Replace empty cells / NaN in all columns with NA
# -----------------------------
df = df.replace("", "NA")   # empty strings → NA
df = df.fillna("NA")        # actual NaN → NA

# -----------------------------
# Clean specific columns: title, tags, one_comment
# -----------------------------
columns_to_clean = ['title', 'tags', 'one_comment']

for col in columns_to_clean:
    if col in df.columns:
        df[col] = df[col].astype(str) \
                            .str.replace(',', ' - ') \
                            .str.replace('\n', ' ') \
                            .str.replace('\r', ' ')

# -----------------------------
# Save final cleaned CSV
# -----------------------------
df.to_csv(output_csv, index=False, lineterminator='\n')

print("✔ Cleaning complete!")
print("Saved to:", output_csv)


✔ Cleaning complete!
Saved to: C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\RhinoVIsulization\Categories\csv\Water_and_mobility_cleaned_final.csv
